# Data acquisition and input preparation

This notebook documents acquisition and preparation of the public TCGA-BRCA input data used by the reported multi-omics subtyping study.

Its role is limited to input construction and quality checks. The final reported predictive analysis is defined in `Preprocessing.ipynb` and uses the fixed-k=50 configuration with 5-fold stratified cross-validation repeated 10 times, shared across all panels and classifiers.

## Data sources

The workflow prepares four molecular layers:
- mRNA expression from the UCSC Xena GDC STAR-TPM distribution;
- gene-level copy-number data from GISTIC2;
- RPPA protein expression;
- somatic mutations and PAM50 subtype annotations from the cBioPortal TCGA PanCancer Atlas distribution.

The mRNA matrix is represented on the `log2(TPM + 1)` scale; no additional logarithmic transform is applied in the reported analysis. Ensembl identifiers are mapped to HUGO gene symbols and the 50 canonical PAM50 genes are checked after reconstruction.

## Cohort construction

Only primary solid-tumour samples are retained, with one sample barcode selected deterministically per patient and layer. Patients must be present across all four layers and have a valid PAM50 label. The resulting complete-case discovery cohort used in the manuscript contains 677 patients.

Intermediate source files may contain more samples than the final complete-case cohort; those intermediate counts are not study results.

## Cross-validation and preprocessing

The shared resampling design is 5 folds × 10 repetitions (50 splits, seed 42). For the reported analysis, all data-dependent preprocessing is fitted within each training fold and then applied to the corresponding test fold. The final representation width is fixed at 50 components per layer.

This notebook may contain input inspection or acquisition helper cells. These cells do not define additional reported analyses, hyperparameter tuning, or nested cross-validation.


**Data directory.** Points the notebook at the local copy of the public input data. Set `TCGA_BRCA_DATA_DIR`; the Colab mount is a fallback.

In [ ]:
# Data directory. Set TCGA_BRCA_DATA_DIR to point at the local copy of the
# public input data; the Google Colab mount below is optional and is skipped
# when the environment variable is already defined.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
if not DATA_DIR.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path(os.environ["TCGA_BRCA_DATA_DIR"])
    except Exception:
        raise SystemExit(
            "Set TCGA_BRCA_DATA_DIR to the directory containing the input data."
        )

**Structure check.** Reads the first rows of the UCSC Xena STAR-TPM matrix to confirm orientation and identifier format before any processing.

In [ ]:
import pandas as pd

path = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'TCGA-BRCA.star_tpm.tsv')

# Quick read of the first 5 rows only, to check the structure
preview = pd.read_csv(path, sep="\t", index_col=0, nrows=5)
print("Shape (preview): ", preview.shape)
print("\nColumns (patients), first: ")
print(preview.columns[:5].tolist())
print("\nIndex (genes), first: ")
print(preview.index[:5].tolist())
print("\nValues (preview): ")
print(preview.iloc[:5, :5])
print(f"\nMin/max over this preview:  {preview.values.min():.3f} / {preview.values.max():.3f}")

**Layer inspection.** Same check for the copy-number, RPPA and clinical files, reporting shape, index and value range.

In [ ]:
import pandas as pd

def strip_ensembl_version(gene_id):
    """ENSG00000000003.15 -> ENSG00000000003"""
    return gene_id.split(".")[0]

def map_ensembl_to_hugo(mrna_df, mapping_file=None):
    """
    Converts versioned Ensembl gene IDs into HUGO symbols.
    mapping_file : table de correspondance (2 colonnes: ensembl_id, hugo_symbol)
                   If absent, attempts to download from mygene or biomart.
    """
    # Step 1: strip version suffixes
    mrna_df = mrna_df.copy()
    mrna_df.index = mrna_df.index.map(strip_ensembl_version)

    if mapping_file is not None:
        mapping = pd.read_csv(mapping_file, sep="\t")
        id_to_symbol = dict(zip(mapping["ensembl_id"], mapping["hugo_symbol"]))
    else:
        # Fast route: the mygene.info API (requires internet, handles batches)
        import mygene
        mg = mygene.MyGeneInfo()
        unique_ids = mrna_df.index.unique().tolist()
        print(f"mygene query for {len(unique_ids)} genes...")
        results = mg.querymany(unique_ids, scopes="ensembl.gene",
                               fields="symbol", species="human")
        id_to_symbol = {r["query"]: r.get("symbol") for r in results if "symbol" in r}

    mrna_df["hugo_symbol"] = mrna_df.index.map(id_to_symbol)
    n_before = mrna_df.shape[0]
    mrna_df = mrna_df.dropna(subset=["hugo_symbol"])
    n_after = mrna_df.shape[0]
    print(f"Mapping : {n_after}/{n_before} genes converted to HUGO symbols "
          f"({n_before - n_after} perdus, sans correspondance)")

    mrna_df = mrna_df.set_index("hugo_symbol")
    mrna_df = mrna_df.groupby(mrna_df.index).mean()  # fusionne les doublons
    return mrna_df

**Dependency.** Installs `mygene`, used to map Ensembl identifiers to HUGO symbols.

In [ ]:
!pip install mygene --quiet

**Identifier survey.** Counts the unique versioned Ensembl identifiers to be mapped.

In [ ]:
import pandas as pd
from pathlib import Path

# Resolve the local input path from TCGA_BRCA_DATA_DIR.
path = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'TCGA-BRCA.star_tpm.tsv')

def patient_id(x):
    p = str(x).split("-")
    return "-".join(p[:3]) if len(p) >= 3 else str(x)

def read_matrix(path, transpose=True):
    df = pd.read_csv(path, sep="\t", index_col=0)
    if transpose:
        df = df.T
    df.index = df.index.map(patient_id)
    return df.groupby(df.index).mean()

print("Lecture du fichier complet (peut prendre 1-2 min, ~900 MB)...")
mrna = read_matrix(path)
print(f"Shape : {mrna.shape}  (patients x genes)")
print(mrna.iloc[:5, :5])

**Ensembl to HUGO mapping.** Strips version suffixes and queries mygene.info in batches; identifiers mapping to the same symbol are merged.

In [ ]:
!pip install mygene --quiet

import mygene
import numpy as np
import time

def strip_ensembl_version(gene_id):
    return gene_id.split(".")[0]

# Step 1: strip version suffixes
gene_cols = mrna.columns.tolist()
stripped = [strip_ensembl_version(g) for g in gene_cols]
unique_ids = sorted(set(stripped))
print(f"{len(unique_ids)} unique Ensembl identifiers to map")

# Step 2: batched queries to avoid timeouts and overload
mg = mygene.MyGeneInfo()
BATCH_SIZE = 2000
id_to_symbol = {}

t0 = time.time()
for i in range(0, len(unique_ids), BATCH_SIZE):
    batch = unique_ids[i:i + BATCH_SIZE]
    try:
        results = mg.querymany(batch, scopes="ensembl.gene",
                               fields="symbol", species="human",
                               verbose=False)
        for r in results:
            if "symbol" in r:
                id_to_symbol[r["query"]] = r["symbol"]
    except Exception as e:
        print(f"  Batch {i}-{i+BATCH_SIZE} failed:  {e}")
    print(f"  Batch {i//BATCH_SIZE + 1}/{(len(unique_ids)-1)//BATCH_SIZE + 1} "
          f"complete ({time.time()-t0:.0f}s elapsed, "
          f"{len(id_to_symbol)} mapped so far)")

print(f"\nTotal mapped:  {len(id_to_symbol)}/{len(unique_ids)} "
      f"({time.time()-t0:.0f}s)")

**Mapping quality.** Counts unmapped identifiers and checks that the fifty canonical PAM50 genes are present.

In [ ]:
# Optional mapping-quality inspection: count unmapped identifiers.
mapped_ids = set(id_to_symbol.keys())
unmapped_ids = set(unique_ids) - mapped_ids
print(f"Unmapped:  {len(unmapped_ids)}")
print(f"Exemples : {list(unmapped_ids)[:20]}")

# Verify presence of the 50 PAM50 genes.
PAM50_GENES = [
    "ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20",
    "CDC6","CDH3","CENPF","CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1",
    "FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C","KRT14","KRT17","KRT5",
    "MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC",
    "NAT1","NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1",
    "SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"
]
mapped_symbols = set(id_to_symbol.values())
pam50_present = [g for g in PAM50_GENES if g in mapped_symbols]
pam50_missing = [g for g in PAM50_GENES if g not in mapped_symbols]
print(f"\nPAM50 genes mapped:  {len(pam50_present)}/{len(PAM50_GENES)}")
if pam50_missing:
    print(f"PAM50 MISSINGS : {pam50_missing}")

**Column rewrite.** Applies the mapping to the mRNA matrix columns.

In [ ]:
new_cols = [id_to_symbol.get(g, None) for g in stripped]
n_before = mrna.shape[1]
keep_mask = [c is not None for c in new_cols]

mrna_mapped = mrna.loc[:, keep_mask].copy()
mrna_mapped.columns = [c for c in new_cols if c is not None]

n_after = mrna_mapped.shape[1]
print(f"Columns:  {n_before} -> {n_after} after mapping")

# Merge duplicated HUGO symbols when multiple Ensembl IDs map to the same symbol.
n_unique_symbols = len(set(mrna_mapped.columns))
print(f"Symboles uniques : {n_unique_symbols} "
      f"(merged duplicates:  {n_after - n_unique_symbols})")
mrna_mapped = mrna_mapped.T.groupby(level=0).mean().T
print(f"Shape finale : {mrna_mapped.shape}")

# Verify that all 50 canonical PAM50 genes are present.
present_final = [g for g in PAM50_GENES if g in mrna_mapped.columns]
print(f"\nPAM50 present dans la matrice finale : {len(present_final)}/50")

**Mapping persistence.** Saves the identifier-to-symbol dictionary so the mapping is reproducible without re-querying the API.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))

# Save the Ensembl-to-HUGO mapping used by the preparation workflow.
mapping_path = DATA_DIR / "ensembl_to_hugo_mapping.json"
with open(mapping_path, "w") as f:
    json.dump(id_to_symbol, f)
print(f"Mapping saved ->  {mapping_path} ({len(id_to_symbol)} entries)")

# Save the HUGO-mapped mRNA matrix used as an input to the analysis.
mrna_out = DATA_DIR / "mrna_hugo_mapped.parquet"
mrna_mapped.to_parquet(mrna_out)
print(f"mRNA matrix saved ->  {mrna_out} ({mrna_mapped.shape})")

**mRNA matrix output.** Writes the HUGO-mapped, primary-tumour-only expression matrix used downstream.

In [ ]:
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))

FILES_TO_CHECK = {
    "CNV (GISTIC2)":       "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
    "RPPA":                "RPPA_RBN.gz",
    "Mutations + PAM50":   "brca_tcga_pan_can_atlas_2018.tar.gz",
}

print(f"Directory checked:  {DATA_DIR}\n")
print(f"{'File':<25}{'Present (.gz)':<16}{'Size':<12}{'Decompressed':<14}")
print("-" * 67)

for label, fname in FILES_TO_CHECK.items():
    p_gz = DATA_DIR / fname
    exists_gz = p_gz.exists()
    size_mb = f"{p_gz.stat().st_size / 1e6:.1f} MB" if exists_gz else "—"

    # For .tar.gz archives no single file is extracted here -- only the
    # presence of the archive itself is checked (extraction is done on the fly
    # par tarfile dans le pipeline, comme pour les mutations).
    if fname.endswith(".tar.gz"):
        decompressed_status = "N/A (extracted on the fly)"
    else:
        p_out = p_gz.with_suffix("")  # retire le .gz
        decompressed_status = "Oui" if p_out.exists() else "Non"

    print(f"{label:<25}{str(exists_gz):<16}{size_mb:<12}{decompressed_status:<14}")

print("\nFull directory listing: ")
for f in sorted(DATA_DIR.glob("*")):
    print(f"  {f.name:<55} {f.stat().st_size/1e6:>10.1f} MB")

**Copy-number retrieval.** Downloads the GISTIC2 gene-level scores.

In [ ]:
import urllib.request
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))

URLS = {
    "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz":
        "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
    "RPPA_RBN.gz":
        "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/RPPA_RBN.gz",
}

for fname, url in URLS.items():
    dest = DATA_DIR / fname
    if dest.exists():
        print(f"Already present:  {fname}")
        continue
    print(f"Downloading {fname} from {url} ...")
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  -> {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"  FAILED:  {e}")

**Copy-number check.** Verifies the downloaded matrix shape and identifiers.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))

for label, fname in [("CNV", "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz"),
                     ("RPPA", "RPPA_RBN.gz")]:
    path = DATA_DIR / fname
    print(f"\n=== {label} ===")
    preview = pd.read_csv(path, sep="\t", index_col=0, compression="gzip", nrows=5)
    print(f"Shape (preview):  {preview.shape}")
    print(f"Columns (patients), first:  {preview.columns[:3].tolist()}")
    print(f"Index (genes/proteins), first:  {preview.index[:5].tolist()}")
    print(preview.iloc[:5, :3])
    print(f"Min/max over the preview:  {preview.values.min():.3f} / {preview.values.max():.3f}")

**Archive inventory.** Lists the files present in the data directory with their sizes.

In [ ]:
import subprocess

# Retrieve the public cBioPortal datahub repository needed for METABRIC input files.
result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/cBioPortal/datahub.git",
     "/content/datahub_tmp"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-2000:])
print("STDERR:", result.stderr[-2000:])

**Archive inventory, continued.** Same listing for the extracted subdirectories.

In [ ]:
import subprocess

# Check Git LFS availability for the public data repository.
r0 = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
print("git-lfs version:", r0.stdout, r0.stderr)

**Presence check.** Confirms that each expected input file is present before preprocessing.

In [ ]:
from pathlib import Path

src = Path("/content/datahub_tmp/public/brca_tcga_pan_can_atlas_2018")
print(f"Dossier existe : {src.exists()}\n")

if src.exists():
    files = sorted(src.glob("*"))
    print(f"{len(files)} files found :\n")
    for f in files:
        size = f.stat().st_size
        print(f"  {f.name:<55} {size:>12,} octets")
else:
    print("Directory not found -- check the clone path")

**Archive handling.** Rebuilds the cBioPortal tar.gz archive; its contents are extracted on the fly rather than unpacked.

In [ ]:
import tarfile
from pathlib import Path

src = Path("/content/datahub_tmp/public/brca_tcga_pan_can_atlas_2018")
dest_archive = Path(str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'brca_tcga_pan_can_atlas_2018.tar.gz'))

print("Rebuilding the .tar.gz archive (may take 1-2 min, ~900 MB total)...")
with tarfile.open(dest_archive, "w:gz") as tar:
    tar.add(src, arcname="brca_tcga_pan_can_atlas_2018")

print(f"-> {dest_archive} ({dest_archive.stat().st_size/1e6:.1f} MB)")

**Archive verification.** Confirms the archive can be opened and lists its members.

In [ ]:
import tarfile

with tarfile.open(dest_archive, "r:gz") as tar:
    names = tar.getnames()
    check_files = ["brca_tcga_pan_can_atlas_2018/data_mutations.txt",
                  "brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"]
    for f in check_files:
        print(f"{f:<60} {'found' if f in names else 'MISSING'}")

**METABRIC retrieval.** Clones the cBioPortal datahub at depth 1 and fetches the `brca_metabric` directory, flagging unresolved LFS pointers.

In [ ]:
"""
=============================================================================
MULTI-OMICS REDUNDANCY / COMPLEMENTARITY PIPELINE
TCGA-BRCA : somatic mutations, CNV, mRNA, RPPA  ->  PAM50
=============================================================================
"""

# %% ========================================================================
# 0. SETUP, CONFIG, PROVENANCE
# ===========================================================================
import os, sys, json, pickle, time, gc, warnings, hashlib, platform, tarfile, shutil
from datetime import datetime
from pathlib import Path
from itertools import combinations
from collections import Counter
from math import factorial

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV)
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon, spearmanr
from xgboost import XGBClassifier
import sklearn, xgboost, scipy, statsmodels

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    if not os.path.exists(str(DATA_DIR)):
        drive.mount("/content/drive")
    DRIVE_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))   # 
except Exception:
    DRIVE_DIR = Path("./TCGA_BRCA_data")

RAW_DIR = DRIVE_DIR / "raw"
PROC_DIR = DRIVE_DIR                        # <-- les .gz sont directement ici, pas dans /raw
PROC_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = os.environ.get("RUN_ID") or pd.Timestamp.now().strftime("%Y-%m-%d")
RUN_DIR = DRIVE_DIR / "runs" / RUN_ID
CKPT_DIR, OUT_DIR = RUN_DIR / "checkpoints", RUN_DIR / "results"
for d in (CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "cv_folds": 5,
    "cv_repeats": 10,
    "seed": 42,
    "filter_stat": "mad",
    "filter_top_k": 5000,
    "min_variance": 1e-6,
    "mut_freq_lo": 0.01,
    "mut_freq_hi": 0.99,
    "rppa_max_missing": 0.20,
    "rppa_knn_k": 5,
    "dim_mode": "fixed_k",
    "dim_k": 50,
    "fdr_alpha": 0.05,
}
SEED = CONFIG["seed"]

import copy as _copy
CONFIG_CANONICAL = _copy.deepcopy(CONFIG)


class cfg_override:
    def __init__(self, **kw):
        self.kw, self.saved = kw, {}
    def __enter__(self):
        for k, v in self.kw.items():
            self.saved[k] = CONFIG[k]
            CONFIG[k] = v
        return CONFIG
    def __exit__(self, *exc):
        CONFIG.update(self.saved)
        return False

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {"mutations": "Mutations", "cnv": "CNV", "mrna": "mRNA", "rppa": "RPPA"}
BINARY_LAYERS = {"mutations"}

RECIPE = {
    "mutations": (False, False, True, False),
    "cnv":       (True,  True,  False, False),
    "mrna":      (True,  True,  False, False),
    "rppa":      (True,  False, False, True),
}

NONSYNONYMOUS = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins", "Splice_Site",
    "Nonstop_Mutation", "Translation_Start_Site",
}

ENV = {"python": sys.version.split()[0], "platform": platform.platform(),
       "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
       "scipy": scipy.__version__, "statsmodels": statsmodels.__version__}
INPUTS = {}
LOG = []
VAR_MISS = {}


class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return o.isoformat(timespec="seconds")
        if isinstance(o, Path):
            return str(o)
        return super().default(o)


def log(msg):
    print(msg)
    LOG.append(f"{datetime.now().strftime('%H:%M:%S')}  {msg}")


def ckpt_save(name, data):
    with open(CKPT_DIR / f"ckpt_{name}.pkl", "wb") as f:
        pickle.dump({"data": data, "run": RUN_ID, "config": CONFIG_CANONICAL,
                     "env": ENV, "inputs": INPUTS,
                     "written": pd.Timestamp.now().isoformat(timespec="seconds")}, f)


def ckpt_load(name):
    p = CKPT_DIR / f"ckpt_{name}.pkl"
    if not p.exists():
        return None
    with open(p, "rb") as f:
        pl = pickle.load(f)
    if isinstance(pl, dict) and "run" in pl:
        if pl["config"] != CONFIG_CANONICAL:
            log(f"  [ckpt] {name} WAS WRITTEN UNDER A DIFFERENT CONFIG -- ignoring it")
            return None
        log(f"  [ckpt] {name} reloaded")
        return pl["data"]
    return pl


MARKER = DRIVE_DIR / "runs" / "_INCOMPLETE"
if "RUN_ID" not in os.environ and MARKER.exists():
    prev = MARKER.read_text().strip()
    if prev and prev != RUN_ID:
        log("!" * 72)
        log(f"An INCOMPLETE run exists: {prev}")
        log(f'    import os; os.environ["RUN_ID"] = "{prev}"')
        log("!" * 72)
MARKER.parent.mkdir(parents=True, exist_ok=True)
MARKER.write_text(RUN_ID)

log("=" * 72)
log(f"RUN_ID {RUN_ID}   ->  {RUN_DIR}")
log("=" * 72)


# %% ========================================================================
# 1. RAW DATA -- public input files
# ===========================================================================
def patient_id(x):
    p = str(x).split("-")
    return "-".join(p[:3]) if len(p) >= 3 else str(x)


def read_matrix_gz(path, transpose=True):
    df = pd.read_csv(path, sep="\t", index_col=0, compression="gzip")
    if transpose:
        df = df.T
    df.index = df.index.map(patient_id)
    return df.groupby(df.index).mean()


MRNA_MAPPED_OVERRIDE = DRIVE_DIR / "mrna_hugo_mapped.parquet"

RAW = {}

# ---- mRNA: use the precomputed HUGO mapping ----
if MRNA_MAPPED_OVERRIDE.exists():
    log(f"  [override] mrna loaded from the precomputed HUGO mapping:  {MRNA_MAPPED_OVERRIDE}")
    RAW["mrna"] = pd.read_parquet(MRNA_MAPPED_OVERRIDE)
else:
    RAW["mrna"] = read_matrix_gz(DRIVE_DIR / "TCGA-BRCA.star_tpm.tsv.gz")

# ---- CNV ----
RAW["cnv"] = read_matrix_gz(DRIVE_DIR / "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz")

# ---- RPPA ----
RAW["rppa"] = read_matrix_gz(DRIVE_DIR / "RPPA_RBN.gz")

# ---- Mutations + PAM50 (from the rebuilt tar.gz archive) ----
tar_path = DRIVE_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
with tarfile.open(tar_path, "r:gz") as tar:
    maf = pd.read_csv(tar.extractfile("brca_tcga_pan_can_atlas_2018/data_mutations.txt"),
                      sep="\t", comment="#", low_memory=False)
    maf = maf[maf["Variant_Classification"].isin(NONSYNONYMOUS)]
    maf["pid"] = maf["Tumor_Sample_Barcode"].map(patient_id)
    mut = (maf.groupby(["pid", "Hugo_Symbol"]).size()
              .unstack(fill_value=0).clip(upper=1).astype(np.int8))
    RAW["mutations"] = mut

    clin = pd.read_csv(tar.extractfile("brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"),
                       sep="\t", comment="#", low_memory=False)

col = "SUBTYPE" if "SUBTYPE" in clin.columns else None
assert col, f"no SUBTYPE column; available: {list(clin.columns)[:25]}"
lab = clin.set_index("PATIENT_ID")[col].dropna().astype(str)
lab = lab.str.replace("^BRCA_", "", regex=True)
lab = lab[lab.isin(["LumA", "LumB", "Her2", "Basal", "Normal"])]
lab.index = lab.index.map(patient_id)
lab.name = "PAM50"
labels_all = lab

# ---- Intersection of patients common to all layers ----
common = set(labels_all.index)
for df in RAW.values():
    common &= set(df.index)
common = sorted(common)
RAW = {k: v.loc[common] for k, v in RAW.items()}
labels = labels_all.loc[common]

le = LabelEncoder()
y = le.fit_transform(labels.values)
N, K = len(y), len(le.classes_)

log(f"\nCohort: {N} patients with complete data across {len(LAYERS)} layers")
log(f"Classes: {dict(zip(le.classes_, np.bincount(y).tolist()))}")
for k in LAYERS:
    log(f"  {SHORT[k]:<12}{str(RAW[k].shape):>16}")
assert np.bincount(y).min() >= CONFIG["cv_folds"], \
    "a class has fewer members than the number of folds"

print("\n✅ Module 0-1 complete: data loaded and aligned.")

**Cohort intersection.** Restricts every layer to primary tumours and to the patients common to all four layers with a valid PAM50 label, producing the 677-patient cohort.

In [ ]:
# %% ========================================================================
# 2. SHARED CROSS-VALIDATION SPLITS
# ===========================================================================
NF, NR = CONFIG["cv_folds"], CONFIG["cv_repeats"]
SPLITS = list(RepeatedStratifiedKFold(n_splits=NF, n_repeats=NR,
                                      random_state=SEED).split(np.zeros(N), y))
REPEAT_OF = [i // NF for i in range(len(SPLITS))]
log(f"\n{len(SPLITS)} splits = {NR} repeats x {NF} folds "
    f"(classifier seed = repeat index)")


# %% ========================================================================
# 3. SINGLE TRAINING-FOLD PREPROCESSING FUNCTION
# ===========================================================================
def score_features(V_df, stat):
    if stat == "mad":
        med = V_df.median()
        return (V_df - med).abs().median()
    if stat == "iqr":
        q = V_df.quantile([0.25, 0.75])
        return q.loc[0.75] - q.loc[0.25]
    if stat == "var":
        return V_df.var()
    raise ValueError(stat)


def reduce_fit(Vtr, layer, seed):
    Reducer = TruncatedSVD if layer in BINARY_LAYERS else PCA
    if CONFIG["dim_mode"] == "fixed_k":
        k = min(CONFIG["dim_k"], Vtr.shape[1] - 1, Vtr.shape[0] - 1)
        return Reducer(n_components=k, random_state=seed).fit(Vtr), k
    kmax = min(CONFIG["dim_k_max"], Vtr.shape[1] - 1, Vtr.shape[0] - 1)
    rd = Reducer(n_components=kmax, random_state=seed).fit(Vtr)
    cum = np.cumsum(rd.explained_variance_ratio_)
    k = int(np.searchsorted(cum, CONFIG["dim_variance_target"]) + 1)
    if cum[-1] < CONFIG["dim_variance_target"]:
        m = VAR_MISS.setdefault(layer, {"n": 0, "min": 1.0, "max": 0.0, "kmax": kmax})
        m["n"] += 1
        m["min"], m["max"] = min(m["min"], cum[-1]), max(m["max"], cum[-1])
        if m["n"] == 1:
            log(f"    [{layer}] variance target {CONFIG['dim_variance_target']:.2f} "
                f"NOT reachable within {kmax} components (reached {cum[-1]:.3f}). Capped.")
    k = min(k, kmax)
    rd.components_ = rd.components_[:k]
    rd.explained_variance_ratio_ = rd.explained_variance_ratio_[:k]
    if hasattr(rd, "explained_variance_"):
        rd.explained_variance_ = rd.explained_variance_[:k]
    if hasattr(rd, "singular_values_"):
        rd.singular_values_ = rd.singular_values_[:k]
    rd.n_components = rd.n_components_ = k
    return rd, k


def preprocess_layer(layer, tr, te, source=None, seed=SEED):
    z_flag, topk_flag, freq_flag, knn_flag = RECIPE[layer]
    df = (RAW if source is None else source)[layer]
    A, B = df.iloc[tr], df.iloc[te]

    if knn_flag:
        miss = A.isna().mean()
        keep = miss[miss <= CONFIG["rppa_max_missing"]].index
        A, B = A[keep], B[keep]
        if A.isna().any().any() or B.isna().any().any():
            imp = KNNImputer(n_neighbors=CONFIG["rppa_knn_k"]).fit(A.values)
            A = pd.DataFrame(imp.transform(A.values), index=A.index, columns=A.columns)
            B = pd.DataFrame(imp.transform(B.values), index=B.index, columns=B.columns)
    elif df.isna().any().any():
        med = A.median()
        A, B = A.fillna(med), B.fillna(med)

    if freq_flag:
        fr = A.mean()
        keep = fr[(fr >= CONFIG["mut_freq_lo"]) & (fr <= CONFIG["mut_freq_hi"])].index
        A, B = A[keep], B[keep]

    va = A.var()
    keep = va[va > CONFIG["min_variance"]].index
    A, B = A[keep], B[keep]

    if topk_flag and A.shape[1] > CONFIG["filter_top_k"]:
        sc = score_features(A, CONFIG["filter_stat"])
        A, B = A[sc.nlargest(CONFIG["filter_top_k"]).index], B[sc.nlargest(CONFIG["filter_top_k"]).index]

    n_feat, Atr, Bte = A.shape[1], A.values.astype(np.float64), B.values.astype(np.float64)

    if z_flag:
        sca = StandardScaler().fit(Atr)
        Atr, Bte = sca.transform(Atr), sca.transform(Bte)

    rd, k = reduce_fit(Atr, layer, seed)
    return (rd.transform(Atr).astype(np.float32),
            rd.transform(Bte).astype(np.float32),
            n_feat, float(rd.explained_variance_ratio_.sum()), k)


# ---- FIXED-k=50 REPRESENTATION CACHE ------------------------------
cache = ckpt_load("cache") or {"red": {}, "var": {}, "nfeat": {}, "ncomp": {}}
RED, VARK, NFEAT, NCOMP = cache["red"], cache["var"], cache["nfeat"], cache["ncomp"]
t0 = time.time()
for s, (tr, te) in enumerate(SPLITS):
    if all((s, k) in RED for k in LAYERS):
        continue
    for k in LAYERS:
        Xtr, Xte, nf, vr, nc = preprocess_layer(k, tr, te)
        RED[(s, k)] = (Xtr, Xte)          #  : tuple, not two separate targets
        VARK[(s, k)] = vr
        NFEAT[(s, k)] = nf
        NCOMP[(s, k)] = nc
    if (s + 1) % 5 == 0 or s == len(SPLITS) - 1:
        ckpt_save("cache", {"red": RED, "var": VARK, "nfeat": NFEAT, "ncomp": NCOMP})
        log(f"  cache {s+1}/{len(SPLITS)}  ({time.time()-t0:.0f}s)")

log("\nPer-layer representation (train-fitted, across folds):")
REPR_SUMMARY = {}
for k in LAYERS:
    nf = [NFEAT[(s, k)] for s in range(len(SPLITS))]
    nc = [NCOMP[(s, k)] for s in range(len(SPLITS))]
    vr = [VARK[(s, k)] for s in range(len(SPLITS))]
    REPR_SUMMARY[SHORT[k]] = {"features_median": int(np.median(nf)),
                              "components_median": int(np.median(nc)),
                              "variance_mean": round(float(np.mean(vr)), 4)}
    log(f"{SHORT[k]:<12}feat={int(np.median(nf))}  comp={int(np.median(nc))}  "
        f"var={np.mean(vr)*100:.1f}%")

print("Module 2-3 complete: reduction cache built.")